## Setup

In [1]:
!pip install imutils
!pip install image-classifiers==1.0.0b1

  Created wheel for imutils: filename=imutils-0.5.3-py3-none-any.whl size=25850 sha256=c511d4c931c6a3d67af271fe5fc357c78ad08fd77c926d797f20d475e29add1d
  Stored in directory: /root/.cache/pip/wheels/fc/9c/6d/1826267c72afa51b564c9c6e0f66abc806879338bc593a2270
Successfully built imutils
  Created wheel for image-classifiers: filename=image_classifiers-1.0.0b1-py3-none-any.whl size=19954 sha256=21326328392d0b1c01e9de2f828216e1819940ac06d79ce7b5979e9704bae5ba
  Stored in directory: /root/.cache/pip/wheels/62/1d/1d/d551ddb7ef02acac3373cb39ccd101661f28635a0d91febb69
Successfully built image-classifiers


In [2]:
# import the necessary packages
import tensorflow as tf
import gc
from tensorflow.keras.optimizers import SGD,RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import Dropout, GlobalAveragePooling2D, Activation, BatchNormalization, Dropout, LSTM, ConvLSTM2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Input,Conv2D, SeparableConv2D, MaxPool2D, LeakyReLU, Activation, LSTM, ConvLSTM2D, Lambda, Reshape, BatchNormalization, Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,TensorBoard,TerminateOnNaN, LearningRateScheduler
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TerminateOnNaN
from tensorflow.keras.layers import Lambda, Reshape, DepthwiseConv2D, ZeroPadding2D, Add, MaxPooling2D,Activation, Flatten, Conv2D, Dense, Input, Dropout, Concatenate, GlobalMaxPooling2D, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
from tensorflow.keras import backend as K

from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, auc
from imutils import paths
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
import shutil
import cv2
import os
from datetime import datetime
%load_ext tensorboard

In [3]:
# from __future__ import absolute_import, division, print_function, unicode_literals
# print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
# tf.config.experimental.list_physical_devices('GPU')
# tf.debugging.set_log_device_placement(True)

## Build Dataset

In [4]:
dataset_path = './dataset'
log_path = './logs'

In [5]:
%%bash
rm -rf dataset
mkdir -p dataset/covid
mkdir -p dataset/normal
mkdir -p dataset/pneumonia
mkdir -p out
mkdir -p logs

### Covid xray dataset

In [6]:
##../input/augmented-covid19-datasets
len(os.listdir('../input/augmented-covid19-datasets/Augmented-COVID-19'))

912

In [7]:
covid_dataset_input_path1 = '../input/augmented-covid19-datasets/Augmented-COVID-19'
covid_dataset_output_path = '../working/dataset/covid'
src_files = os.listdir(covid_dataset_input_path1)
for file_name in src_files:
    full_file_name = os.path.join(covid_dataset_input_path1, file_name)
    if os.path.isfile(full_file_name):
        shutil.copy(full_file_name, covid_dataset_output_path)

In [8]:
len(os.listdir('../working/dataset/covid'))

912

In [9]:
## chest-xray-covid19-pneumonia
len(os.listdir('../input/chest-xray-covid19-pneumonia/Data/test/COVID19'))

116

In [10]:
covid_dataset_input_path2 = '../input/chest-xray-covid19-pneumonia/Data/test/COVID19'
covid_dataset_output_path = '../working/dataset/covid'
src_files = os.listdir(covid_dataset_input_path2)
for file_name in src_files:
    full_file_name = os.path.join(covid_dataset_input_path2, file_name)
    if os.path.isfile(full_file_name):
        shutil.copy(full_file_name, covid_dataset_output_path)

In [11]:
len(os.listdir('../working/dataset/covid'))

1028

In [12]:
len(os.listdir('../input/chest-xray-covid19-pneumonia/Data/train/COVID19'))

460

In [13]:
covid_dataset_input_path3 = '../input/chest-xray-covid19-pneumonia/Data/train/COVID19'
covid_dataset_output_path = '../working/dataset/covid'
src_files = os.listdir(covid_dataset_input_path3)
for file_name in src_files:
    full_file_name = os.path.join(covid_dataset_input_path3, file_name)
    if os.path.isfile(full_file_name):
        shutil.copy(full_file_name, covid_dataset_output_path)

In [14]:
len(os.listdir('../working/dataset/covid'))

1488

In [15]:
## covid19-sirm-database
len(os.listdir('../input/covid19-sirm-database/COVID-19'))

37

In [16]:
covid_dataset_input_path4 = '../input/covid19-sirm-database/COVID-19'
covid_dataset_output_path = '../working/dataset/covid'
src_files = os.listdir(covid_dataset_input_path4)
for file_name in src_files:
    full_file_name = os.path.join(covid_dataset_input_path4, file_name)
    if os.path.isfile(full_file_name):
        shutil.copy(full_file_name, covid_dataset_output_path)

In [17]:
len(os.listdir('../working/dataset/covid'))

1525

In [18]:
covid_dataset_input_path5 = '../input/covid19-chest-xray-image-repository/covid19'
covid_dataset_output_path = '../working/dataset/covid'
src_files = os.listdir(covid_dataset_input_path5)
for file_name in src_files:
    full_file_name = os.path.join(covid_dataset_input_path5, file_name)
    if os.path.isfile(full_file_name):
        shutil.copy(full_file_name, covid_dataset_output_path)

In [19]:
len(os.listdir('../working/dataset/covid'))

2272

In [20]:
covid_dataset_input_path6 = '../input/covid19imagerepositoryfrom/covid-19-image-repository-master'
covid_dataset_output_path = '../working/dataset/covid'
csvPath = os.path.sep.join([covid_dataset_input_path6, "data.csv"])
df = pd.read_csv(csvPath)

for (i, row) in df.iterrows():
    if row["projection"] != "pa":
        continue

    imagePath = os.path.sep.join([covid_dataset_input_path6, "png", (row["image_id"]+".png")])
#     print(imagePath)

    if not os.path.exists(imagePath):
        continue

    filename = (row["image_id"]+".png")
    outputPath = os.path.sep.join([f"{dataset_path}/covid", filename])
#     print(outputPath)

    shutil.copy2(imagePath, outputPath)

In [21]:
len(os.listdir('../working/dataset/covid'))

2313

### Build pneumonia xray dataset

In [22]:
## NIH Chest X-rays
len(os.listdir('../input/data/images_012/images'))

7121

In [23]:
csvPath = '../input/data/Data_Entry_2017.csv'
df = pd.read_csv(csvPath)

for (i, row) in df.iterrows():
    # if (1) the current case is not COVID-19 or (2) this is not
    # a 'PA' view, then ignore the row
    if row["Finding Labels"] != "Pneumonia" or row["View Position"] != "PA":
        continue

    if os.path.exists(os.path.sep.join(['../input/data/images_001', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_001', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_002', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_002', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_003', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_003', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_004', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_004', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_005', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_005', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_006', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_006', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_007', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_007', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_008', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_008', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_009', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_009', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_010', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_010', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_011', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_011', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    elif os.path.exists(os.path.sep.join(['../input/data/images_012', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_012', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
    else:
        continue


In [24]:
len(os.listdir('../working/dataset/pneumonia'))

176

In [25]:
## covid19 = 2313
samples = 2137

In [26]:
pneumonia_dataset_path ='../input/chest-xray-pneumonia/chest_xray'

In [27]:
basePath = os.path.sep.join([pneumonia_dataset_path, "train", "PNEUMONIA"])
imagePaths = list(paths.list_images(basePath))

# randomly sample the image paths
random.seed(42)
random.shuffle(imagePaths)
imagePaths = imagePaths[:samples]

for (i, imagePath) in enumerate(imagePaths):
    filename = imagePath.split(os.path.sep)[-1]
    outputPath = os.path.sep.join([f"{dataset_path}/pneumonia", filename])

    shutil.copy2(imagePath, outputPath)

In [28]:
len(os.listdir('../working/dataset/pneumonia'))

2313

### Build normal xray dataset

In [29]:
csvPath = '../input/data/Data_Entry_2017.csv'
df = pd.read_csv(csvPath)
count = 0

for (i, row) in df.iterrows():
    # if (1) the current case is not COVID-19 or (2) this is not
    # a 'PA' view, then ignore the row
    if row["Finding Labels"] != "No Finding" or row["View Position"] != "PA":
        continue

    if os.path.exists(os.path.sep.join(['../input/data/images_001', "images", row["Image Index"]])):
        imagePath = os.path.sep.join(['../input/data/images_001', "images", row["Image Index"]])
        filename = row["Image Index"].split(os.path.sep)[-1]
        outputPath = os.path.sep.join([f"{dataset_path}/normal", filename])
        shutil.copy2(imagePath, outputPath)
#         print(outputPath)
        count += 1
    else:
        continue
        
    if count == 2313:
        break


In [30]:
len(os.listdir('../working/dataset/normal'))

1977

In [31]:
## covid19 = 2313
samples = 336

In [32]:
basePath = os.path.sep.join([pneumonia_dataset_path, "train", "NORMAL"])
imagePaths = list(paths.list_images(basePath))

# randomly sample the image paths
random.seed(42)
random.shuffle(imagePaths)
imagePaths = imagePaths[:samples]

for (i, imagePath) in enumerate(imagePaths):
    filename = imagePath.split(os.path.sep)[-1]
    outputPath = os.path.sep.join([f"{dataset_path}/normal", filename])

    shutil.copy2(imagePath, outputPath)

In [33]:
len(os.listdir('../working/dataset/normal'))

2313

In [34]:
!ls ../working/dataset/

covid  normal  pneumonia


In [35]:
dir_name = '../working/dataset/'

In [36]:
shutil.make_archive('COVID19_Pneumonia_Normal_Chest_Xray_PA_Dataset', 'zip', dir_name)

'/kaggle/working/COVID19_Pneumonia_Normal_Chest_Xray_PA_Dataset.zip'

In [37]:
!ls ../working/

COVID19_Pneumonia_Normal_Chest_Xray_PA_Dataset.zip  dataset  out
__notebook__.ipynb				    logs


In [38]:
!rm -rf dataset
!rm -rf logs
!rm -rf out